In [4]:
import sys
import polars as pl
from pathlib import Path
import json

ROOT = Path().resolve()

# поднимаемся до проекта (а не src!)
while not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from ingestion.tmdb import extract_tmdb_raw

In [12]:
def normalize_tmdb(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .select([
            "tmdb_id",
            "title",
            "original_title",
            "overview",
            "release_date",
            "vote_average",
            "vote_count",
            "popularity",
            "original_language",
            "adult",
            "poster_path",
            "backdrop_path",
            "genre_ids",
            "source"
        ])
        .with_columns([
            # --- CLEAN STRINGS ---
            pl.col("title").fill_null("").str.strip_chars(),
            pl.col("original_title").fill_null("").str.strip_chars(),
            pl.col("overview").fill_null("").str.strip_chars(),

            # --- SAFE YEAR EXTRACTION ---
            pl.col("release_date")
              .cast(pl.Utf8, strict=False)
              .str.slice(0, 4)
              .cast(pl.Int32, strict=False)
              .alias("year"),


            # --- TYPE NORMALIZATION ---
            pl.col("vote_average").cast(pl.Float64, strict=False),
            pl.col("vote_count").cast(pl.Int32, strict=False),
            pl.col("popularity").cast(pl.Float64, strict=False),

            # --- BOOLEAN SAFETY ---
            pl.col("adult").cast(pl.Boolean, strict=False),
        ])
    )



In [7]:
df_raw = extract_tmdb_raw()


In [13]:
df_clean = normalize_tmdb(df_raw)


In [14]:
df_clean.write_parquet("tmdb.parquet")

In [15]:
df_clean.show()

tmdb_id,title,original_title,overview,release_date,vote_average,vote_count,popularity,original_language,adult,poster_path,backdrop_path,genre_ids,source,year
i64,str,str,str,str,f64,i32,f64,str,bool,str,str,list[i64],str,i32
1198994,"""Send Help""","""Send Help""","""Two colleagues become stranded…","""2026-01-22""",7.0,1054,309.1211,"""en""",false,"""/mjkS2iAgWj3ik1DTjvI15nHZ7yl.j…","""/gCmfeKmEAZBP5gcXpiqb0gii9rS.j…","[27, 53, 35]","""tmdb""",2026
1470130,"""The Mortuary Assistant""","""The Mortuary Assistant""","""Rebecca Owens, a recent mortua…","""2026-02-13""",5.481,77,198.9187,"""en""",false,"""/72AoFPC5TY4DfJwXXS9rPwPeReD.j…","""/gM1kQRPwOVW1Eos5tVWqcYmHR5S.j…","[27, 9648]","""tmdb""",2026
1290417,"""Thrash""","""Thrash""","""When a Category 5 hurricane de…","""2026-04-10""",5.975,387,153.9028,"""en""",false,"""/adk8weka3O5648g3de4z3y4aE7G.j…","""/3ooXDVaz4xHKtwe4lkmF1gNopOC.j…","[27, 53]","""tmdb""",2026
1304313,"""Lee Cronin's The Mummy""","""Lee Cronin's The Mummy""","""The young daughter of a journa…","""2026-04-15""",6.908,125,166.7205,"""en""",false,"""/8L8efNkz8rUmwR7sV0g3vnC9yjn.j…","""/zc08qkHtWfZ2cQGQBK1S5yafJmR.j…","[27, 9648]","""tmdb""",2026
1010755,"""The Strangers: Chapter 3""","""The Strangers: Chapter 3""","""Tethered by a frightening conc…","""2026-02-05""",5.672,116,131.2846,"""en""",false,"""/yPHwX78mcwJw3I6YOJ9qh2wQBFr.j…","""/fPfDB0IHOqvmKIzFP9CA51OOS1N.j…","[27, 53]","""tmdb""",2026
